In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel, Kernel
from numpy.typing import NDArray
import numpy as np
from typing import Sequence
import matplotlib.pyplot as plt 


In [ ]:
var_f = 2.0
var_n = 2.5
ls = 1.2

kernel = (
    ConstantKernel(constant_value=var_f)
    * RBF(length_scale=ls) + WhiteKernel(noise_level=var_n)
)

def exp_kernel(A: NDArray, B: NDArray, ls: float) -> NDArray:
    # RBF kernel: k(xa, xb) = exp(-0.5 * ||xa - xb||^2) (lengthscale=1)
    A = np.asarray(A)  # (n_a, n_features)
    B = np.asarray(B)  # (n_b, n_features)

    # squared euclidean distance matrix (n_a, n_b)
    sqdist = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)

    return np.exp(-0.5 * sqdist / (ls**2))


In [ ]:
n_dim = 3
n_per_dim = 5

x = np.linspace(-1, 1, n_per_dim)
g = np.meshgrid(*([x]*n_dim))

# Create the full grid and reshape to (n_samples, 3)
X = np.vstack([gi.ravel() for gi in g]).T  # (25, 3)

gram_A = kernel(X, X)
gram_B = var_f*exp_kernel(X, X, ls)#  + var_n * np.eye(n_per_dim**n_dim)


fig, axs = plt.subplots(1, 3)

axs[0].imshow(gram_A, cmap='viridis', aspect="auto")
axs[0].set_title("scipy SE kernel")
axs[1].imshow(gram_B, cmap='viridis', aspect="auto")
axs[1].set_title("Homemade SE kernel")
axs[2].imshow(gram_B- gram_A, cmap='viridis', aspect="auto")
axs[2].set_title("Difference")
plt.tight_layout()
plt.show()

### Hilbert Space Gaussian Process

In [ ]:
type ArrayLike = NDArray | Sequence[int | float]

def power_spectral_density(omega: NDArray, ls: float | NDArray, n_dims: int) -> NDArray:
    """
    Power spectral density for the ExpQuad kernel.

    .. math::

        S(\\boldsymbol\\omega) =
            (\\sqrt(2 \\pi)^D \\prod_{i}^{D}\\ell_i
            \\exp\\left( -\\frac{1}{2} \\sum_{i}^{D}\\ell_i^2 \\omega_i^{2} \\right)

    Args:
        omega: array of shape (m_star, d), frequencies at which to evaluate the PSD.
        ls:    lengthscale(s), either a scalar or array of shape (d,).
        n_dims: number of input dimensions D.

    Returns:
        Array of shape (m_star,), one PSD value per basis function.
    """
    ls_arr = np.ones(n_dims) * ls          # (d,)
    c = np.power(np.sqrt(2.0 * np.pi), n_dims)
    exp = np.exp(-0.5 * np.dot(np.square(omega), np.square(ls_arr)))  # (m_star,)
    return c * np.prod(ls_arr) * exp       # (m_star,)

# def calc_eigenvalues(L: NDArray, m: Sequence[int]) -> NDArray:
#     """Calculate eigenvalues of the Laplacian."""
#     temp = [np.arange(1, 1 + m[d]) for d in range(len(m))]
#     S = np.meshgrid(*temp)
#     S_arr = np.vstack([s.flatten() for s in S]).T
#     print(S)

#     return np.square((np.pi * S_arr) / (2 * L))


# # def calc_eigenvectors(Xs: NDArray, L: NDArray, eigvals: NDArray, m: Sequence[int]) -> NDArray:
# #     """Calculate eigenvectors of the Laplacian.

# #     These are used as basis vectors in the HSGP approximation.

# #     Parameters 
# #     --------
# #     Xs : NDArray, shape (n_samples, n_features)
# #     L : NDArray, shape (n_features, )
# #     eigvals
# #     """
# #     m_star = int(np.prod(m))
# #     phi = np.ones((Xs.shape[0], m_star))
# #     for d in range(len(m)):
# #         c = 1.0 / np.sqrt(L[d])
# #         term1 = np.sqrt(eigvals[:, d])
# #         term2 = np.tile(Xs[:, d][:, None], m_star) + L[d]
# #         phi *= c * np.sin(term1 * term2)
# #     return phi

# def calc_eigenvectors(Xs: NDArray, L: NDArray, eigvals: NDArray) -> NDArray:
#     """Calculate eigenvectors of the Laplacian.
#     These are used as basis vectors in the HSGP approximation.

#     Parameters
#     ----------
#     Xs      : NDArray, shape (n_samples, d)
#     L       : NDArray, shape (d,)
#     eigvals : NDArray, shape (m_star, d)
#     m       : Sequence[int], shape (d,)

#     Returns
#     -------
#     phi : NDArray, shape (n_samples, m_star)
#     """
#     # (1, m_star, d) * (n_samples, 1, d) -> (n_samples, m_star, d)
#     term1 = np.sqrt(eigvals)[None, :, :]          # (1, m_star, d)
#     term2 = Xs[:, None, :] + L[None, None, :]     # (n_samples, 1, d) + (1, 1, d)
#     c = 1.0 / np.sqrt(L)                          # (d,)

#     phi = c * np.sin(term1 * term2)               # (n_samples, m_star, d)
#     return np.prod(phi, axis=-1)                  # (n_samples, m_star)

def calc_eigenvalues(L: ArrayLike, m: int, d: int) -> NDArray:
    """
    Calculate eigenvalues of the Laplacian on [-L1,L1] x ... x [-Ld,Ld]
    with Dirichlet boundary conditions, returning the m smallest.

    Parameters
    ----------
    L : NDArray, shape (d,)
        Domain half-widths per dimension.
    m : int
        Number of eigenfunctions to return.
    d : int
        Number of input dimensions.

    Returns
    -------
    selected_per_dim_eigenvalues : NDArray, shape (m,d)
        The m smallest eigenvalues, sorted ascending.
    """
    L = np.asarray(L, dtype=float)
    
    # Number of indices per dimension, scaled by relative domain size
    N_per_dim = np.ceil(m ** (1 / d) * L / L.min()).astype(int)

    # Build full multi-index grid (Cartesian product of per-dim indices)
    temp = [np.arange(1, 1 + N_per_dim[dim]) for dim in range(d)]
    grids = np.meshgrid(*temp, indexing='ij')
    NN = np.vstack([g.ravel() for g in grids]).T      # (N_total, d)

    # Compute all eigenvalues
    per_dim_eigvals = np.square((np.pi * NN) / (2 * L)) # (N_total, d)
    all_eigenvalues = np.sum(per_dim_eigvals, axis=1)

    # Sort and keep the m smallest
    sort_idx = np.argsort(all_eigenvalues)[:m]
    selected_per_dim_eigenvalues = per_dim_eigvals[sort_idx]
    # NN = NN[sort_idx]

    return selected_per_dim_eigenvalues

def calc_eigenvectors(Xs: NDArray, L: ArrayLike, per_dim_eigvals: NDArray) -> NDArray:
    """Calculate eigenvectors of the Laplacian.
    These are used as basis vectors in the HSGP approximation.

    Parameters
    ----------
    Xs              : NDArray, shape (n_samples, d)
    L               : NDArray, shape (d,)
    per_dim_eigvals : NDArray, shape (m, d)
        Per dimension eigenvalues, with the smallest sum

    Returns
    -------
    phi : NDArray, shape (n_samples, m)
    """
    L = np.asarray(L, dtype=float)
    
    # (1, m, d) * (n_samples, 1, d) -> (n_samples, m, d)
    term1 = np.sqrt(per_dim_eigvals)[None, :, :]          # (1, m, d)
    term2 = Xs[:, None, :] + L[None, None, :]     # (n_samples, 1, d) + (1, 1, d)
    c = 1.0 / np.sqrt(L)                          # (d,)

    phi = c * np.sin(term1 * term2)               # (n_samples, m, d)
    return np.prod(phi, axis=-1)                  # (n_samples, m)

In [ ]:
X =np.asarray([
    [0.5, 1.0],
    [-0.3, 0.8]
]) 
m = 50
L = np.asarray([1.0, 1.5])

eigvals = calc_eigenvalues(L, m, d=2)
print(eigvals)
eigvecs = calc_eigenvectors(X, L, eigvals)
print(eigvecs)

In [ ]:
n_dim = 3
bound = 1
m_hat = 60
ls = 1.0
sigma_f = 5
sigma_n = 1e-1
extension = 2
rng = np.random.default_rng(seed=0)

# Define grid points along each axis
n_per_dim = 5

x = np.linspace(-1, 1, n_per_dim)
g = np.meshgrid(*([x]*n_dim))

# Create the full grid and reshape to (n_samples, 3)
X = np.vstack([gi.ravel() for gi in g]).T  # (25, 3)
# X = 2*rng.random((125,3))
# X = rng.normal(0, (1,1,.1), size=(125,3))
L = np.ones(n_dim, dtype=float)*extension
m = 1000
# m = [25, 25, 25]


In [ ]:

eigvals = calc_eigenvalues(L, m, n_dim)  # (m_start, d)
print(f"{eigvals.shape = }")


In [ ]:

phi = calc_eigenvectors(X, L, eigvals)  # (n_samples, m_star)
print(phi.shape)
omega = np.sqrt(eigvals)  # (m_star, d)
print(omega.shape)


In [ ]:

psd = power_spectral_density(omega, ls, n_dim)*sigma_f**2
print(psd.shape)


In [ ]:

gram_C = (phi * psd) @ phi.T # + np.eye(X.shape[0])*sigma_n**2
A = phi.T @ phi

In [ ]:
rng = np.random.default_rng(seed=0)

# X = 2*bound*rng.random((100, n_dim)) - bound

gram_B = sigma_f**2*exp_kernel(X, X, ls) # + np.eye(X.shape[0])*sigma_n**2


In [ ]:

fig, axs = plt.subplots(1, 3)

axs[0].imshow(gram_B, cmap='viridis', aspect="auto")
axs[0].set_title("Exact Gram Matrix")
axs[1].imshow(gram_C, cmap='viridis', aspect="auto")
axs[1].set_title("Approximate HSGP")
axs[2].imshow(gram_B - gram_C, cmap='viridis', aspect="auto")
axs[2].set_title("Difference")
plt.tight_layout()
plt.show()
print(np.max(gram_B - gram_C))

In [ ]:
N = 1000
F_norms = np.empty(N)
gram_B = sigma_f**2*exp_kernel(X, X, ls) # + np.eye(X.shape[0])*sigma_n**2

for m in range(0,N) :
    eigvals = calc_eigenvalues(L, m, n_dim)  # (m_start, d)
    phi = calc_eigenvectors(X, L, eigvals)  # (n_samples, m_star)
    omega = np.sqrt(eigvals)  # (m_star, d)
    psd = power_spectral_density(omega, ls, n_dim)*sigma_f**2
    gram_C = (phi * psd) @ phi.T # + np.eye(X.shape[0])*sigma_n**2
    F_norms[m] = np.linalg.norm(gram_C - gram_B, 'fro')


fig, axs = plt.subplots(1, 1)
axs.plot(np.log(F_norms))
plt.tight_layout()
plt.show()  
